# 02 - Preprocessing

This notebook prepares the UCI Heart Disease dataset for later binary classification modeling.

The work here is limited to preprocessing: target creation, train/test splitting, missing-value imputation, one-hot encoding, validation, and saving processed data.

## 1. Imports and Paths

The raw dataset is loaded from `../data/raw/heart_disease_uci.csv`, and processed outputs are written under `../data/processed/`.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
RAW_DATA_PATH = Path("../data/raw/heart_disease_uci.csv")
PROCESSED_DATA_DIR = Path("../data/processed")

# Keep the notebook runnable if it is executed from the project root instead of notebooks/.
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = Path("data/raw/heart_disease_uci.csv")
    PROCESSED_DATA_DIR = Path("data/processed")

RAW_DATA_PATH

WindowsPath('../data/raw/heart_disease_uci.csv')

## 2. Load the Raw Data

The raw file contains the original `num` disease severity label. This notebook converts that multiclass severity label into a binary classification target.

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    str    
 3   dataset   920 non-null    str    
 4   cp        920 non-null    str    
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    str    
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    str    
 13  ca        309 non-null    float64
 14  thal      434 non-null    str    
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(2), str(6)
memory usage: 115.1+ KB


## 3. Create the Binary Target

The original `num` column records disease severity. For binary classification, any severity above zero is treated as heart disease:

- `target = 0` when `num == 0`
- `target = 1` when `num > 0`

After creating `target`, both `id` and `num` are excluded from the modeling features. `id` is only an identifier, and `num` would leak the label into the feature matrix.

In [5]:
df = df.copy()
df["target"] = (df["num"] > 0).astype(int)

df[["num", "target"]].head()

,num,target
0,0,0
1,2,1
2,1,1
3,0,0
4,0,0


In [6]:
df["target"].value_counts(normalize=True).rename("proportion")

target
1    0.553261
0    0.446739
Name: proportion, dtype: float64

## 4. Define Features and Label

The feature matrix `X` contains only model-eligible predictors. The label vector `y` contains the new binary target.

In [7]:
X = df.drop(columns=["id", "num", "target"])
y = df["target"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (920, 14)
y shape: (920,)


## 5. Define Column Groups

Numerical features will be median-imputed. Categorical features will be imputed with the most frequent value and then one-hot encoded.

In [8]:
numerical_columns = ["age", "trestbps", "chol", "thalch", "oldpeak"]
categorical_columns = [
    "sex",
    "dataset",
    "cp",
    "fbs",
    "restecg",
    "exang",
    "slope",
    "ca",
    "thal",
]

expected_columns = numerical_columns + categorical_columns

missing_columns = sorted(set(expected_columns) - set(X.columns))
unexpected_columns = sorted(set(X.columns) - set(expected_columns))

assert not missing_columns, f"Missing expected columns: {missing_columns}"
assert not unexpected_columns, f"Unexpected feature columns: {unexpected_columns}"

X = X[expected_columns]
X.head()

,age,trestbps,chol,thalch,oldpeak,sex,dataset,cp,fbs,restecg,exang,slope,ca,thal
0,63,145.0,233.0,150.0,2.3,Male,Cleveland,typical angina,True,lv hypertrophy,False,downsloping,0.0,fixed defect
1,67,160.0,286.0,108.0,1.5,Male,Cleveland,asymptomatic,False,lv hypertrophy,True,flat,3.0,normal
2,67,120.0,229.0,129.0,2.6,Male,Cleveland,asymptomatic,False,lv hypertrophy,True,flat,2.0,reversable defect
3,37,130.0,250.0,187.0,3.5,Male,Cleveland,non-anginal,False,normal,False,downsloping,0.0,normal
4,41,130.0,204.0,172.0,1.4,Female,Cleveland,atypical angina,False,lv hypertrophy,False,upsloping,0.0,normal


## 6. Train/Test Split

The split uses stratification so the binary target distribution is similar in the train and test sets.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (736, 14)
X_test shape: (184, 14)
y_train shape: (736,)
y_test shape: (184,)


In [10]:
pd.DataFrame(
    {
        "train": y_train.value_counts(normalize=True).sort_index(),
        "test": y_test.value_counts(normalize=True).sort_index(),
    }
)

,train,test
target,,
0,0.447011,0.445652
1,0.552989,0.554348


## 7. Build the Preprocessing Pipeline

The preprocessor has separate branches for numerical and categorical columns:

- Numerical columns: median imputation
- Categorical columns: most-frequent imputation followed by one-hot encoding

The encoder ignores categories that appear later but were not seen during training.

In [11]:
try:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    one_hot_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", one_hot_encoder),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

## 8. Fit on Training Data Only

The preprocessor is fit only on `X_train`. The fitted transformations are then applied to both `X_train` and `X_test` to avoid using test-set information during preprocessing.

In [12]:
X_train_processed_array = preprocessor.fit_transform(X_train)
X_test_processed_array = preprocessor.transform(X_test)

print(f"Processed X_train shape: {X_train_processed_array.shape}")
print(f"Processed X_test shape: {X_test_processed_array.shape}")

Processed X_train shape: (736, 32)
Processed X_test shape: (184, 32)


## 9. Convert Processed Arrays to DataFrames

The fitted preprocessor provides the expanded feature names after one-hot encoding. Those names are used to convert the processed arrays back into pandas DataFrames.

In [13]:
feature_names = preprocessor.get_feature_names_out()

if hasattr(X_train_processed_array, "toarray"):
    X_train_processed_array = X_train_processed_array.toarray()
if hasattr(X_test_processed_array, "toarray"):
    X_test_processed_array = X_test_processed_array.toarray()

X_train_processed = pd.DataFrame(
    X_train_processed_array,
    columns=feature_names,
    index=X_train.index,
)

X_test_processed = pd.DataFrame(
    X_test_processed_array,
    columns=feature_names,
    index=X_test.index,
)

X_train_processed.head()

,num__age,num__trestbps,num__chol,num__thalch,num__oldpeak,cat__sex_Female,cat__sex_Male,cat__dataset_Cleveland,cat__dataset_Hungary,cat__dataset_Switzerland,...,cat__slope_downsloping,cat__slope_flat,cat__slope_upsloping,cat__ca_0.0,cat__ca_1.0,cat__ca_2.0,cat__ca_3.0,cat__thal_fixed defect,cat__thal_normal,cat__thal_reversable defect
640,53.0,160.0,0.0,122.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
743,74.0,130.0,0.0,140.0,0.5,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
890,53.0,124.0,243.0,122.0,2.0,0.0,1.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
270,61.0,140.0,207.0,138.0,1.9,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
654,56.0,155.0,0.0,99.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


## 10. Validate Missing Values

After imputation and encoding, the processed train and test feature sets should contain no missing values.

In [14]:
missing_value_check = pd.Series(
    {
        "X_train_processed_missing": int(X_train_processed.isna().sum().sum()),
        "X_test_processed_missing": int(X_test_processed.isna().sum().sum()),
        "y_train_missing": int(y_train.isna().sum()),
        "y_test_missing": int(y_test.isna().sum()),
    }
)

missing_value_check

X_train_processed_missing    0
X_test_processed_missing     0
y_train_missing              0
y_test_missing               0
dtype: int64

In [15]:
assert (missing_value_check == 0).all(), missing_value_check

## 11. Save Processed Data

The processed features and labels are saved as CSV files for future modeling notebooks.

In [16]:
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

X_train_processed.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_test_processed.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)
y_train.to_frame(name="target").to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
y_test.to_frame(name="target").to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)

sorted(path.name for path in PROCESSED_DATA_DIR.glob("*.csv"))

['X_test.csv',
 'X_train.csv',
 'heart_processed.csv',
 'y_test.csv',
 'y_train.csv']

## 12. Summary

This notebook created a leakage-safe binary target, split the data with stratification, fit preprocessing only on the training features, transformed train and test features consistently, verified that no missing values remain, and saved the clean processed datasets for later modeling.